In [48]:
pip install biopython beautifulsoup4 pandas lxml requests

In [49]:
from Bio import Entrez
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

Entrez.email = "your_email@example.com"


# ----------------------------
# PubMed 검색
# ----------------------------
def search_pubmed(protein, max_results=20):

    query = f"{protein} western blot"

    handle = Entrez.esearch(
        db="pubmed",
        term=query,
        retmax=max_results
    )

    record = Entrez.read(handle)

    return record["IdList"]


# ----------------------------
# PMID -> PMCID
# ----------------------------
def get_pmcid(pmid):

    url = f"https://www.ncbi.nlm.nih.gov/pmc/utils/idconv/v1.0/?ids={pmid}&format=json"

    r = requests.get(url).json()

    try:
        return r["records"][0]["pmcid"]
    except:
        return None


# ----------------------------
# PMC XML 다운로드
# ----------------------------
def download_xml(pmcid):

    url = f"https://www.ncbi.nlm.nih.gov/pmc/articles/{pmcid}/?format=xml"

    r = requests.get(url)

    if r.status_code == 200:
        return r.text

    return None


# ----------------------------
# figure caption 추출
# ----------------------------
def extract_figure_captions(xml_text):

    soup = BeautifulSoup(xml_text, "lxml")

    captions = []

    for fig in soup.find_all("fig"):

        cap = fig.find("caption")

        if cap:
            captions.append(cap.get_text(" ", strip=True))

    return captions


# ----------------------------
# 본문 문장 추출
# ----------------------------
def extract_body_sentences(xml_text):

    soup = BeautifulSoup(xml_text, "lxml")

    sentences = []

    for p in soup.find_all("p"):

        text = p.get_text(" ", strip=True)

        split = re.split(r'\.|\n', text)

        sentences.extend(split)

    return sentences


# ----------------------------
# western blot 필터
# ----------------------------
def filter_wb_sentences(sentences):

    wb_sentences = []

    for s in sentences:

        s_lower = s.lower()

        if any(k in s_lower for k in [
            "western blot",
            "immunoblot",
            "wb analysis"
        ]):

            wb_sentences.append(s.strip())

    return wb_sentences


# ----------------------------
# effect detection
# ----------------------------
def detect_effect(sentence):

    s = sentence.lower()

    if any(w in s for w in [
        "increase",
        "upregulate",
        "elevated",
        "enhanced"
    ]):
        return "increase"

    if any(w in s for w in [
        "decrease",
        "reduce",
        "downregulate",
        "suppressed"
    ]):
        return "decrease"

    return "unknown"


# ----------------------------
# phospho detection
# ----------------------------
def detect_phospho(sentence):

    s = sentence.lower()

    if any(k in s for k in [
        "phosphorylation",
        "phospho",
        "p-"
    ]):
        return "phospho"

    return "total"


# ----------------------------
# gene knockdown detection
# ----------------------------
def detect_knockdown(sentence):

    s = sentence.lower()

    if any(k in s for k in [
        "knockdown",
        "sirna",
        "shrna",
        "silencing"
    ]):
        return True

    return False


# ----------------------------
# drug detection (간단 버전)
# ----------------------------
def detect_drug(sentence):

    drug_list = [
        "ly294002",
        "wortmannin",
        "rapamycin",
        "torin",
        "mk2206"
    ]

    s = sentence.lower()

    for d in drug_list:

        if d in s:
            return d

    return None


# ----------------------------
# 메인 파이프라인
# ----------------------------
def run_pipeline(protein, max_results=20):

    pmids = search_pubmed(protein, max_results)

    results = []

    for pmid in pmids:

        pmcid = get_pmcid(pmid)

        if not pmcid:
            continue

        xml = download_xml(pmcid)

        if not xml:
            continue

        captions = extract_figure_captions(xml)
        body = extract_body_sentences(xml)

        sentences = captions + body

        wb_sentences = filter_wb_sentences(sentences)

        for s in wb_sentences:

            if protein.lower() not in s.lower():
                continue

            effect = detect_effect(s)

            phospho = detect_phospho(s)

            kd = detect_knockdown(s)

            drug = detect_drug(s)

            results.append({

                "PMID": pmid,
                "protein": protein,
                "form": phospho,
                "drug": drug,
                "gene_KD": kd,
                "effect": effect,
                "sentence": s

            })

    return pd.DataFrame(results)


# ----------------------------
# 실행
# ----------------------------

df = run_pipeline("AKT", max_results=30)

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
print(df.head())

df.to_csv("western_blot_results.csv", index=False)

       PMID protein     form  drug  gene_KD    effect  \
0  41829957     AKT  phospho  None    False   unknown   
1  41829957     AKT    total  None    False   unknown   
2  41829039     AKT  phospho  None    False   unknown   
3  41829039     AKT  phospho  None    False  increase   
4  41828681     AKT  phospho  None    False  increase   

                                                                                                                                                                                                                                                                                                       sentence  
0  Western blot analysis showed no significant effect on calmodulin-dependent Protein Kinase Kinase 2 (CaMKK2) expression but did reveal a modest yet statistically significant elevation in phosphorylated calcium/calmodulin-dependent protein kinase IV (pCaMKIV) and phosphorylated AKT (pAKT) ( Figure 5 )  
1                                            